In [1]:
import gc
import os
import sys

import numpy as np
import optuna
import pandas as pd
import torch
from sklearn.model_selection import KFold

current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, ".."))
sys.path.append(parent_dir)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
%load_ext autoreload
%autoreload 2
from drGAT import No_atten_drGAT
from drGAT.load_data import load_data
from drGAT.sampler import RandomSampler

In [6]:
drugAct, pos_num, null_mask, S_d, S_c, S_g, A_cg, A_dg = load_data("gdsc2")

load gdsc2
unique drugs: 69
unique genes: 153
DTI unique genes:  153
Top 90% variable genes:  1957
Total:  2089
Final gene exp shape: (910, 2089)
Final drug Act shape: (240, 910)


/Users/inouey2/miniconda3/envs/torch/lib/python3.10/site-packages/numpy/core/fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
25it [00:08,  2.86it/s]


Done!


In [7]:
PATH = "../gdsc2_data/"

In [10]:
def objective(trial):
    params = {
        "n_drug": S_d.shape[0],
        "n_cell": S_c.shape[0],
        "n_gene": S_g.shape[0],
        "dropout1": trial.suggest_categorical("dropout1", [0.1, 0.2, 0.3, 0.4, 0.5]),
        "dropout2": trial.suggest_categorical("dropout2", [0.1, 0.2, 0.3, 0.4, 0.5]),
        "hidden1": trial.suggest_categorical(
            "hidden1",
            [256, 512, 1028],
        ),
        "hidden2": trial.suggest_categorical(
            "hidden2",
            [
                128,
                256,
                512,
            ],
        ),
        "hidden3": trial.suggest_categorical(
            "hidden3",
            [
                64,
                128,
                256,
            ],
        ),
        "epochs": trial.suggest_categorical("epochs", [10, 50, 100, 200, 500]),
        # "epochs": 2,
        "heads": trial.suggest_categorical("heads", [1, 2, 3, 4, 5]),
        "activation": trial.suggest_categorical(
            "activation", ["relu", "gelu", "swish"]
        ),
        "optimizer": trial.suggest_categorical("optimizer", ["Adam", "AdamW", "SGD"]),
        "lr": trial.suggest_float("lr", 1e-5, 1e-2, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True),
        "scheduler": trial.suggest_categorical(
            "scheduler", [None, "Cosine", "Step", "Plateau"]
        ),
        "gnn_layer": trial.suggest_categorical(
            "gnn_layer",
            ["GCN", "MPNN"],
        ),
    }

    # スケジューラ関連パラメータの条件付き追加
    if params["scheduler"] == "Cosine":
        params["T_max"] = trial.suggest_int("T_max", 20, 50)
    elif params["scheduler"] == "Step":
        params["scheduler_gamma"] = trial.suggest_float("gamma_step", 0.1, 0.95)
        params["step_size"] = trial.suggest_int("step_size", 10, 30)
    elif params["scheduler"] == "Plateau":
        params["patience"] = trial.suggest_int("patience_plateau", 3, 10)
        params["threshold"] = trial.suggest_float(
            "thresh_plateau", 1e-4, 1e-2, log=True
        )

    if params["hidden1"] < params["hidden2"] or params["hidden2"] < params["hidden3"]:
        raise optuna.TrialPruned("Invalid layer size configuration")

    if params["optimizer"] in ["Adam", "AdamW"]:
        params["amsgrad"] = trial.suggest_categorical("amsgrad", [True, False])

    if params["optimizer"] == "SGD":
        params["momentum"] = trial.suggest_float("momentum", 0.8, 0.95)
        params["nesterov"] = trial.suggest_categorical("nesterov", [True, False])

    # 隠れ層サイズとバッチサイズの関係を制約
    if (params["hidden1"] > 512) and (params["hidden2"] > 256):
        raise optuna.TrialPruned("Memory intensive configuration")

    try:
        k = 5
        kfold = KFold(n_splits=k, shuffle=True, random_state=42)

        res = pd.DataFrame()
        for train_index, test_index in kfold.split(np.arange(pos_num)):
            sampler = RandomSampler(
                drugAct,
                train_index,
                test_index,
                null_mask,
                S_d,
                S_c,
                S_g,
                A_cg,
                A_dg,
                PATH,
            )
            _, best_metrics, _ = No_atten_drGAT.train(
                sampler, params=params, device=device, verbose=False
            )

            res = pd.concat(
                [
                    res,
                    pd.DataFrame(best_metrics, index=["acc", "f1", "auroc", "aupr"]).T,
                ]
            )

        return [float(i) for i in res.mean()]

    except RuntimeError as e:
        if "CUDA out of memory" in str(e):
            print(f"Pruned trial {trial.number}: CUDA OOM")

            # メモリ解放処理
            with torch.cuda.device("cuda"):
                torch.cuda.empty_cache()
            gc.collect()

            # Pruning通知
            raise optuna.TrialPruned(f"OOM at trial {trial.number}")

        else:
            raise e

In [12]:
name = "GDSC2"
study = optuna.create_study(
    directions=["maximize"] * 4,
    sampler=optuna.samplers.TPESampler(),
    pruner=optuna.pruners.HyperbandPruner(),
    storage="sqlite:///{}_{}.sqlite3".format(name, "GCN_MPNN"),
    study_name=name,
    load_if_exists=True,
)
study.optimize(objective, n_trials=100, n_jobs=8)

[I 2025-03-28 12:41:17,453] A new study created in RDB with name: GDSC2
[I 2025-03-28 12:41:18,169] Trial 5 pruned. Invalid layer size configuration
[I 2025-03-28 12:41:18,208] Trial 6 pruned. Invalid layer size configuration
[I 2025-03-28 12:41:18,303] Trial 0 pruned. Memory intensive configuration


Using device: cpu
Using device: cpu


[I 2025-03-28 12:41:27,019] Trial 3 pruned. Memory intensive configuration


Using device: cpu
Using device: cpu


/Users/inouey2/miniconda3/envs/torch/lib/python3.10/site-packages/torch/amp/grad_scaler.py:132: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  warnings.warn(
  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]


  0%|          | 0/2 [00:00<?, ?it/s][I 2025-03-28 12:41:27,673] Trial 9 pruned. Invalid layer size configuration
[I 2025-03-28 12:41:27,737] Trial 10 pruned. Invalid layer size configuration
[I 2025-03-28 12:41:27,756] Trial 11 pruned. Memory intensive configuration
[I 2025-03-28 12:41:28,414] Trial 8 pruned. Invalid layer size configuration
[I 2025-03-28 12:41:28,520] Trial 12 pruned. Invalid layer size configuration
[I 2025-03-28 12:41:28,911] Trial 15 pruned. Memory intensive configuration
[W 2025-03-28 12:41:28,945] Trial 17 failed with parameters: {} because of the following error: TypeError("'NoneType' object is not iterable").
Traceback (most recent call last):
  File "/Users/inouey2/miniconda3/envs/t

...







 50%|█████     | 1/2 [00:26<00:26, 26.41s/it]





100%|██████████| 2/2 [00:37<00:00, 18.54s/it]


Best model found at epoch 2


/Users/inouey2/miniconda3/envs/torch/lib/python3.10/site-packages/torch/amp/grad_scaler.py:132: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  warnings.warn(


Using device: cpu








  0%|          | 0/2 [00:00<?, ?it/s]

## Eval

In [ ]:
# test_drug = test_data.values[:, 0]
# test_cell = test_data.values[:, 1]

# test_labels = np.load("data/test_labels.npy")
# test_labels = torch.tensor(test_labels).float()
# test = [drug, cell, gene, edge_index, test_drug, test_cell, test_labels]

In [ ]:
# prob, res, test_attention = drGAT.eval(model, test)